# Noise analysis: idling and measurement

**Premise.** How often should syndrome measurements happen? The question has an answer
only when two costs move in opposite directions and can be varied independently:

```
  wait longer   ->  idling error accumulates
  measure more  ->  each round costs measurement error
```

**One question: does Selene give independent control of those two?** Everything else is
held at zero.

The notebook climbs a ladder, because the answer changes as it goes:

| Section | Memory | Gates present | Separation |
| --- | --- | --- | --- |
| 2 | one physical qubit | none | exact |
| 3 | — | CNOTs, examined directly | — |
| 4 | Steane \|0_L> | encoder + extraction | differential |

## 0. Setup

In [1]:
from collections import Counter

from guppylang import guppy
from guppylang.std.angles import angle
from guppylang.std.builtins import array, output, py
from guppylang.std.quantum import collect_measurements, cx, h, measure, measure_array, qubit, x, z
from guppylang.std.qsystem import phased_x, rz

from selene_sim import DepolarizingErrorModel, Quest, build

N_DATA = 7
N_QUBITS = 16

_BUILT = {}


def runner(key, make):
    """Compile once per key. build() dominates the runtime of every sweep here."""
    if key not in _BUILT:
        _BUILT[key] = build(make().compile())
    return _BUILT[key]


def run(key, make, n_shots=600, seed=1, **channels):
    """Run a program under a model carrying only the named channels.

    Keep n_shots in the hundreds: very small shot counts trip a Selene stream error.
    """
    return [dict(s) for s in runner(key, make).run_shots(
        simulator=Quest(), n_qubits=N_QUBITS, n_shots=n_shots,
        error_model=DepolarizingErrorModel(random_seed=seed, **channels),
        random_seed=seed)]

## 1. The knobs

```
DepolarizingErrorModel(random_seed, p_1q, p_2q, p_meas, p_init)
SimpleLeakageErrorModel(random_seed, p_leak, leak_measurement_bias)
IdealErrorModel(random_seed)
```

Every channel defaults to zero, so naming one isolates it.

**Every parameter attaches to an operation, and duration appears nowhere.** Durations live
in `SimpleRuntime(duration_ns_rxy, duration_ns_measure, ...)`, a separate object feeding
scheduling rather than error.

So **a qubit left alone accrues error at rate zero**, and idling has to be written into the
circuit as explicit operations.

In [2]:
@guppy
def bare_wait() -> None:
    q = qubit()
    output("m", measure(q).read())


for label, ch in [("gate channels at 0.3", dict(p_1q=0.3, p_2q=0.3)), ("ideal", {})]:
    shots = run("bare_wait", lambda: bare_wait, n_shots=600, **ch)
    print(f"  untouched qubit, {label:22s} -> P(1) = {sum(s['m'] for s in shots) / 600:.3f}")

print("\nan untouched qubit stays perfect: idling is free until written in")

  untouched qubit, gate channels at 0.3   -> P(1) = 0.000
  untouched qubit, ideal                  -> P(1) = 0.000

an untouched qubit stays perfect: idling is free until written in


## 2. The gate-free memory

A memory experiment on one *physical* qubit is prepare, wait, measure. No gates at all, so
this is where the two noise types separate exactly.

### 2a. Measurement noise alone

Does `p_meas` isolate, and does the number mean what it says?

In [3]:
@guppy
def prep_measure() -> None:
    """|0>, measured immediately. Any 1 is a measurement fault."""
    q = qubit()
    output("m", measure(q).read())


print("p_meas   P(read 1)   ratio")
for pm in (0.0, 0.02, 0.05, 0.1, 0.2, 0.4):
    shots = run("prep_measure", lambda: prep_measure, n_shots=2000, p_meas=pm)
    f = sum(s["m"] for s in shots) / 2000
    print(f"{pm:5.2f}   {f:9.4f}   {f / pm if pm else float('nan'):5.2f}")

print("\np_meas isolates, and reads directly as the flip probability")

p_meas   P(read 1)   ratio
 0.00      0.0000     nan
 0.02      0.0215    1.07
 0.05      0.0505    1.01
 0.10      0.0965    0.96
 0.20      0.1985    0.99
 0.40      0.4060    1.01

p_meas isolates, and reads directly as the flip probability


### 2b. Idling noise alone

Selene ships no T1, amplitude-damping or dephasing model — the three error models are
depolarizing, leakage and ideal, and all of them attach to operations. (A custom model is
possible through the `ErrorModel` ABC, which wants a compiled `library_file`, so it means
writing a plugin rather than Python.)

Idling therefore needs an explicit operation, and `p_1q` is the channel that pays for it.
The obvious trick is to insert identity pairs and count them as waiting time. **The
optimiser has an opinion about that**, so each candidate has to be tested: applied 20 times
to a qubit that should stay in `|0>`, with `p_1q` as the only live channel.

In [4]:
def idle_program(kind, n):
    """A qubit held through n idle units of the given kind, then measured."""
    if kind == "phased_x(0,0)":
        @guppy
        def prog() -> None:
            q = qubit()
            for _ in range(py(n)):
                phased_x(q, angle(0.0), angle(0.0))
            output("m", measure(q).read())
    elif kind == "rz(0)":
        @guppy
        def prog() -> None:
            q = qubit()
            for _ in range(py(n)):
                rz(q, angle(0.0))
            output("m", measure(q).read())
    elif kind == "x;x":
        @guppy
        def prog() -> None:
            q = qubit()
            for _ in range(py(n)):
                x(q); x(q)
            output("m", measure(q).read())
    elif kind == "z;z":
        @guppy
        def prog() -> None:
            q = qubit()
            for _ in range(py(n)):
                z(q); z(q)
            output("m", measure(q).read())
    elif kind == "x;z;x;z":
        @guppy
        def prog() -> None:
            q = qubit()
            for _ in range(py(n)):
                x(q); z(q); x(q); z(q)
            output("m", measure(q).read())
    else:
        @guppy
        def prog() -> None:
            q = qubit()
            for _ in range(py(n)):
                h(q); h(q)
            output("m", measure(q).read())
    return prog


print("candidate idle unit   20x at p_1q=0.05   verdict")
for kind in ("phased_x(0,0)", "rz(0)", "x;x", "z;z", "h;h", "x;z;x;z"):
    shots = run(("idle", kind, 20), lambda k=kind: idle_program(k, 20), n_shots=1000, p_1q=0.05)
    f = sum(s["m"] for s in shots) / 1000
    print(f"  {kind:18s}  P(1) = {f:.3f}         {'SURVIVES' if f > 0.01 else 'cancelled'}")

print("\nAdjacent identical gates are peepholed away, so x;x, z;z and h;h all cost nothing.")
print("x;z;x;z survives, because no two neighbours are inverses of each other.")
print("phased_x(0,0) is the cleanest: one idle unit is exactly one charged operation,")
print("which keeps tau linear in the accumulated error.")

candidate idle unit   20x at p_1q=0.05   verdict
  phased_x(0,0)       P(1) = 0.355         SURVIVES
  rz(0)               P(1) = 0.355         SURVIVES
  x;x                 P(1) = 0.000         cancelled
  z;z                 P(1) = 0.000         cancelled
  h;h                 P(1) = 0.000         cancelled
  x;z;x;z             P(1) = 0.496         SURVIVES

Adjacent identical gates are peepholed away, so x;x, z;z and h;h all cost nothing.
x;z;x;z survives, because no two neighbours are inverses of each other.
phased_x(0,0) is the cleanest: one idle unit is exactly one charged operation,
which keeps tau linear in the accumulated error.


In [5]:
# Idling as a function of waiting time, with phased_x as the idle operation.
print(" tau   P(flip)")
for tau in (0, 1, 2, 5, 10, 20, 40):
    shots = run(("idle", "phased_x(0,0)", tau),
                lambda t=tau: idle_program("phased_x(0,0)", t), n_shots=1000, p_1q=0.02)
    print(f"{tau:4d}   {sum(s['m'] for s in shots) / 1000:.3f}")

print("\nerror grows with tau and saturates toward 0.5")
print("on one physical qubit the two channels are exactly separable: p_1q is idling, p_meas is readout")

 tau   P(flip)
   0   0.000
   1   0.014
   2   0.023
   5   0.059
  10   0.109
  20   0.181
  40   0.313

error grows with tau and saturates toward 0.5
on one physical qubit the two channels are exactly separable: p_1q is idling, p_meas is readout


## 3. What a logical memory adds

A memory on a *logical* qubit cannot stay gate-free. Measuring a stabilizer means
entangling the data block with an ancilla and reading the ancilla out, so syndrome
extraction **is** a CNOT circuit — twelve CNOTs per basis per round for Steane. That cost
is the reason the interval question exists: waiting is what avoids paying it.

The gates fall into three groups:

| gates | status |
| --- | --- |
| extraction CNOTs | irreducible — this *is* the measurement |
| idle operations | irreducible — this *is* the waiting |
| encoder, 3 H + 9 CX | one-time preparation, arguably outside the memory |

### The catch

A CNOT compiles to native operations that include single-qubit rotations, and **each of
those is charged `p_1q`** — the same channel idling uses.

In [6]:
def cx_chain(n):
    """Two qubits, n CNOTs, no 1q gates in the source at all."""
    @guppy
    def prog() -> None:
        a, b = qubit(), qubit()
        for _ in range(py(n)):
            cx(a, b)
        output("ma", measure(a).read())
        output("mb", measure(b).read())
    return prog


print("a CX-only circuit, 600 shots")
print("  n_cx   channel        P(any 1)")
for n in (0, 10):
    for ch in ({"p_1q": 0.05}, {"p_2q": 0.05}):
        shots = run(("cx", n), lambda k=n: cx_chain(k), n_shots=600, **ch)
        f = sum(1 for s in shots if s["ma"] or s["mb"]) / 600
        print(f"  {n:4d}   {list(ch)[0]}=0.05     {f:.3f}")

print("\nCNOTs draw on p_1q, and draw on it heavily.")
print("Source-level gate counts say nothing about which channel pays: read the compiled form.")

a CX-only circuit, 600 shots
  n_cx   channel        P(any 1)
     0   p_1q=0.05     0.000
     0   p_2q=0.05     0.000
    10   p_1q=0.05     0.575
    10   p_2q=0.05     0.297

CNOTs draw on p_1q, and draw on it heavily.
Source-level gate counts say nothing about which channel pays: read the compiled form.


## 4. The encoded timestep

```
encode |0_L>  ->  tau idle ops on each data qubit  ->  syndrome_z  ->  read out
```

Only the Z-stabilizers are measured, which keeps the circuit small. The X and Z channels
are independent in a CSS code, so a bit-flip memory answers the interval question.

A clean state gives syndrome `0`, so the fraction of shots with a non-zero syndrome is the
**detection-event rate**.

Because extraction CNOTs and idling share `p_1q`, the circuit has a fixed error floor at
τ=0. Read idling **differentially**: `rate(τ) − rate(0)` removes the floor.

In [7]:
@guppy
def encode_zero() -> array[qubit, 7]:
    qs = array(qubit() for _ in range(7))
    h(qs[0]); h(qs[1]); h(qs[3])
    cx(qs[0], qs[2]); cx(qs[0], qs[4]); cx(qs[0], qs[6])
    cx(qs[1], qs[2]); cx(qs[1], qs[5]); cx(qs[1], qs[6])
    cx(qs[3], qs[4]); cx(qs[3], qs[5]); cx(qs[3], qs[6])
    return qs


@guppy
def syndrome_z(qs: array[qubit, 7]) -> array[bool, 3]:
    a0, a1, a2 = qubit(), qubit(), qubit()
    cx(qs[0], a0); cx(qs[2], a0); cx(qs[4], a0); cx(qs[6], a0)
    cx(qs[1], a1); cx(qs[2], a1); cx(qs[5], a1); cx(qs[6], a1)
    cx(qs[3], a2); cx(qs[4], a2); cx(qs[5], a2); cx(qs[6], a2)
    return array(measure(a0).read(), measure(a1).read(), measure(a2).read())


def timestep(tau):
    @guppy
    def prog() -> None:
        qs = encode_zero()
        for _ in range(py(tau)):
            for i in range(7):
                phased_x(qs[i], angle(0.0), angle(0.0))
        output("sz", syndrome_z(qs))
        output("data", collect_measurements(measure_array(qs)))
    return prog


def event_rate(tau, n_shots=600, **channels):
    """Fraction of shots whose Z-syndrome fires."""
    shots = run(("ts", tau), lambda t=tau: timestep(t), n_shots=n_shots, **channels)
    return sum(1 for s in shots if any(s["sz"])) / n_shots


TAUS = (0, 2, 5, 10, 20)
P1Q, PMEAS = 0.001, 0.01

print(f"p_1q={P1Q} (idling + gates),  p_meas={PMEAS} (readout)\n")
print(" tau   idle only   minus floor   meas only   both")
floor = event_rate(0, p_1q=P1Q)
for tau in TAUS:
    a = event_rate(tau, p_1q=P1Q)
    b = event_rate(tau, p_meas=PMEAS)
    c = event_rate(tau, p_1q=P1Q, p_meas=PMEAS)
    print(f"{tau:4d}   {a:9.3f}   {a - floor:11.3f}   {b:9.3f}   {c:5.3f}")

print(f"\nfloor at tau=0 is {floor:.3f}: the encoder and extraction CNOTs, before any waiting")
print("the 'minus floor' column is the idling cost, and it is the one that grows with tau")
print("measurement noise stays flat in tau, which is what makes the trade-off a trade-off")

p_1q=0.001 (idling + gates),  p_meas=0.01 (readout)

 tau   idle only   minus floor   meas only   both
   0       0.057         0.000       0.017   0.073
   2       0.062         0.005       0.028   0.088
   5       0.077         0.020       0.025   0.098
  10       0.103         0.047       0.040   0.140
  20       0.127         0.070       0.032   0.155

floor at tau=0 is 0.057: the encoder and extraction CNOTs, before any waiting
the 'minus floor' column is the idling cost, and it is the one that grows with tau
measurement noise stays flat in tau, which is what makes the trade-off a trade-off


## Findings

1. **`p_meas` isolates cleanly** and reads directly as the probability of a flipped
   outcome.

2. **Idling is free until written in.** No error model carries a duration, so the idling
   rate is a **declared** parameter rather than a measured one.

3. **`phased_x(q, 0, 0)` is the idle operation.** Adjacent identical gates are peepholed away, so `x;x`, `z;z` and `h;h` cost nothing; `x;z;x;z` survives. `phased_x` keeps one idle unit equal to one charged operation.
   `p_1q`; `rz(q, 0)` behaves identically. `h; h` cancels during compilation.

4. **On one physical qubit the separation is exact.** Gate-free memory gives `p_1q` for
   idling and `p_meas` for readout, independently.

5. **On a logical qubit the separation becomes differential.** CNOTs compile to native
   operations charged at `p_1q`, so extraction and idling share a channel. Subtracting the
   τ=0 floor recovers the idling cost. Keep `p_1q` near 0.001 so the floor stays small
   enough for τ to move the result.

6. **The interval question is answerable**, with the caveat stated plainly: the *shape* of
   the trade-off is measured, while its *calibration* is declared.

### Next

- Sweep τ against round count R to locate the interior optimum.
- Bare against flagged ancillas: the same τ, a different per-round cost (O1).
- `CircuitExtractor` gives per-shot fault attribution via `get_error_model_output()`,
  turning these rates into counts of the operations actually corrupted.